# 用上下文检索增强 RAG

> 说明：如需了解更多关于上下文检索的背景信息，包括在多个数据集上的额外性能评估，建议阅读我们的配套[博客文章](https://www.anthropic.com/news/contextual-retrieval)。

检索增强生成（RAG）让 Claude 在给出响应时能够利用您的内部知识库、代码库或任何其他文档语料。企业越来越多地构建 RAG 应用，以改进客户支持、公司内部文档问答、财务与法律分析、代码生成等工作流。

在另一篇[指南](https://github.com/anthropics/anthropic-cookbook/blob/main/capabilities/retrieval_augmented_generation/guide.ipynb)中，我们讲解了如何搭建一个基础检索系统，演示了如何评估它的性能，并列出了几种提升性能的技术。在本指南中，我们介绍一种提升检索性能的技术：上下文嵌入向量。

在传统 RAG 中，文档通常被切分为较小的块以便高效检索。这种方式对许多应用效果良好，但当单个块缺乏足够上下文时就会出问题。上下文嵌入向量在嵌入之前为每个块添加相关上下文，从而解决这一问题。该方法提升了每个嵌入块的质量，使检索更准确，整体性能也随之提升。在我们测试的所有数据源上取平均，上下文嵌入向量使前 20 块的检索失败率下降 35%。

同样的块级上下文也可以用于 BM25 检索，以进一步提升检索性能。我们在“上下文 BM25”一节中介绍这项技术。

在本指南中，我们将演示如何用 9 个代码库的数据集作为知识库，构建并优化一个上下文检索系统。我们将依次讲解：

1）搭建基础检索工作流，建立性能基线。

2）上下文嵌入向量：它是什么、为什么有效，以及提示词缓存如何让它在生产场景中切实可用。

3）实现上下文嵌入向量，并展示性能提升。

4）上下文 BM25：用*上下文* BM25 混合检索提升性能。

5）用重排序提升性能，

### 评估指标与数据集：

我们使用一个已预先切块的 9 个代码库数据集——这些块均按基础字符切分机制切块。我们的评估数据集包含 248 条查询——每条查询都包含一个“黄金块”。我们将使用名为 Pass@k 的指标来评估性能。Pass@k 检查“黄金文档”是否出现在每条查询检索到的前 k 个文档中。在本例中，上下文嵌入向量帮助我们把 Pass@10 性能从 ~87% --> ~95%。

你可以在 `data/codebase_chunks.json` 中找到代码文件及其块，在 `data/evaluation_set.jsonl` 中找到评估数据集

#### 补充说明：

使用这种检索方法时，提示词缓存有助于控制成本。该功能目前可在 Anthropic 自有 API 上使用，并即将支持 AWS Bedrock 和 GCP Vertex 上的第三方合作伙伴环境。我们知道许多客户在构建 RAG 方案时会使用 AWS Knowledge Bases 和 GCP Vertex AI API，这两种平台经过少量定制都能使用该方法。如需相关指导，请联系 Anthropic 或您的 AWS/GCP 客户团队！

为了便于在 Bedrock 上使用该方法，AWS 团队向我们提供了一段代码，你可以用它实现一个为每个文档添加上下文的 Lambda 函数。部署该 Lambda 函数后，在配置 [Bedrock Knowledge Base](https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base-create.html) 时可以把这个函数选为自定义切块选项。这段代码位于 `contextual-rag-lambda-function` 中，主要的 lambda 函数代码在 `lambda_function.py`。

## 目录

1）环境准备

2）基础 RAG

3）上下文嵌入向量

4）上下文 BM25

5）重排序

## 环境准备

开始本指南之前，请确认您具备：

**技术技能：**
- 中级 Python 编程能力
- 对 RAG（检索增强生成）的基本理解
- 熟悉向量数据库和嵌入向量
- 基本的命令行操作能力

**系统要求：**
- Python 3.8+
- 已安装并运行 Docker（可选，用于 BM25 检索）
- 4GB 以上可用 RAM
- 约 5-10 GB 磁盘空间，用于向量数据库

**API 访问：**
- [Anthropic API 密钥](https://console.anthropic.com/)（免费额度即可满足）
- [Voyage AI API 密钥](https://www.voyageai.com/)
- [Cohere API 密钥](https://cohere.com/)

**时间与成本：**
- 预计完成时间：30-45 分钟
- API 成本：跑完整个数据集约 $5-10

### 库 

我们需要几个库，包括：

1） `anthropic` - 用于与 Claude 交互

2） `voyageai` - 用于生成高质量嵌入向量

3） `cohere` - 用于重排序

4） `elasticsearch` 用于高性能的 BM25 检索

3） `pandas`、`numpy`、`matplotlib` 和 `scikit-learn` 用于数据处理和可视化

### 环境变量 

请确保设置以下环境变量：

```
- VOYAGE_API_KEY
- ANTHROPIC_API_KEY
- COHERE_API_KEY
```

In [6]:
%%capture
%pip install --upgrade anthropic voyageai cohere elasticsearch pandas numpy

我们在开头定义模型名称，以便新模型发布时方便更换模型

In [ ]:
MODEL_NAME = "claude-haiku-4-5"

我们首先初始化 Anthropic 客户端，它将用于生成上下文描述。

In [5]:
import os

import anthropic

client = anthropic.Anthropic(
    # This is the default and can be omitted
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

## 初始化一个向量数据库类

我们创建一个 VectorDB 类来处理嵌入向量存储和相似度检索。该类在我们的 RAG 工作流中承担三项关键功能：

1. **嵌入向量生成**：使用 Voyage AI 的嵌入模型把文本块转换为向量表示
2. **存储与缓存**：把嵌入向量保存到磁盘，避免重新计算（从而节省时间和 API 成本）
3. **相似度检索**：使用余弦相似度检索与某个查询最相关的块


本指南使用一个带 pickle 序列化的简单内存向量数据库。这让代码易于理解，且不需要外部依赖。该类在生成后自动把嵌入向量保存到磁盘，因此嵌入成本只需支付一次。

在生产环境中，请考虑托管的向量数据库方案。

下面的 VectorDB 类遵循与生产方案相同的接口模式，便于日后替换。主要特性包括批处理（每次 128 个块）、用 tqdm 进行进度跟踪，以及查询缓存以加快评估期间的重复检索。

In [ ]:
import json
import pickle
from typing import Any

import numpy as np
import voyageai
from tqdm import tqdm


class VectorDB:
    def __init__(self, name: str, api_key=None):
        if api_key is None:
            api_key = os.getenv("VOYAGE_API_KEY")
        self.client = voyageai.Client(api_key=api_key)
        self.name = name
        self.embeddings = []
        self.metadata = []
        self.query_cache = {}
        self.db_path = f"./data/{name}/vector_db.pkl"

    def load_data(self, dataset: list[dict[str, Any]]):
        if self.embeddings and self.metadata:
            print("Vector database is already loaded. Skipping data loading.")
            return
        if os.path.exists(self.db_path):
            print("Loading vector database from disk.")
            self.load_db()
            return

        texts_to_embed = []
        metadata = []
        total_chunks = sum(len(doc["chunks"]) for doc in dataset)

        with tqdm(total=total_chunks, desc="Processing chunks") as pbar:
            for doc in dataset:
                for chunk in doc["chunks"]:
                    texts_to_embed.append(chunk["content"])
                    metadata.append(
                        {
                            "doc_id": doc["doc_id"],
                            "original_uuid": doc["original_uuid"],
                            "chunk_id": chunk["chunk_id"],
                            "original_index": chunk["original_index"],
                            "content": chunk["content"],
                        }
                    )
                    pbar.update(1)

        self._embed_and_store(texts_to_embed, metadata)
        self.save_db()

        print(f"Vector database loaded and saved. Total chunks processed: {len(texts_to_embed)}")

    def _embed_and_store(self, texts: list[str], data: list[dict[str, Any]]):
        batch_size = 128
        with tqdm(total=len(texts), desc="Embedding chunks") as pbar:
            result = []
            for i in range(0, len(texts), batch_size):
                batch = texts[i : i + batch_size]
                batch_result = self.client.embed(batch, model="voyage-2").embeddings
                result.extend(batch_result)
                pbar.update(len(batch))

        self.embeddings = result
        self.metadata = data

    def search(self, query: str, k: int = 20) -> list[dict[str, Any]]:
        if query in self.query_cache:
            query_embedding = self.query_cache[query]
        else:
            query_embedding = self.client.embed([query], model="voyage-2").embeddings[0]
            self.query_cache[query] = query_embedding

        if not self.embeddings:
            raise ValueError("No data loaded in the vector database.")

        similarities = np.dot(self.embeddings, query_embedding)
        top_indices = np.argsort(similarities)[::-1][:k]

        top_results = []
        for idx in top_indices:
            result = {
                "metadata": self.metadata[idx],
                "similarity": float(similarities[idx]),
            }
            top_results.append(result)

        return top_results

    def save_db(self):
        data = {
            "embeddings": self.embeddings,
            "metadata": self.metadata,
            "query_cache": json.dumps(self.query_cache),
        }
        os.makedirs(os.path.dirname(self.db_path), exist_ok=True)
        with open(self.db_path, "wb") as file:
            pickle.dump(data, file)

    def load_db(self):
        if not os.path.exists(self.db_path):
            raise ValueError(
                "Vector database file not found. Use load_data to create a new database."
            )
        with open(self.db_path, "rb") as file:
            data = pickle.load(file)
        self.embeddings = data["embeddings"]
        self.metadata = data["metadata"]
        self.query_cache = json.loads(data["query_cache"])

现在我们可以用这个类加载数据集

In [12]:
# Load your transformed dataset
with open("data/codebase_chunks.json") as f:
    transformed_dataset = json.load(f)

# Initialize the VectorDB
base_db = VectorDB("base_db")

# Load and process the data
base_db.load_data(transformed_dataset)

Embedding chunks: 100%|██████████| 737/737 [00:42<00:00, 17.28it/s]

Vector database loaded and saved. Total chunks processed: 737


## 基础 RAG

首先，我们用一种最简方式搭建基础 RAG 工作流。业内许多人把这称为“朴素 RAG”。基础 RAG 工作流包括以下 3 个步骤：

1）按标题切分文档——每个块只包含对应子标题下的内容

2）对每个文档做嵌入

3）使用余弦相似度检索文档，以回答查询

In [26]:
import json
from collections.abc import Callable
from typing import Any

from tqdm import tqdm


def load_jsonl(file_path: str) -> list[dict[str, Any]]:
    """Load JSONL file and return a list of dictionaries."""
    with open(file_path) as file:
        return [json.loads(line) for line in file]


def evaluate_retrieval(
    queries: list[dict[str, Any]], retrieval_function: Callable, db, k: int = 20
) -> dict[str, float]:
    total_score = 0
    total_queries = len(queries)

    for query_item in tqdm(queries, desc="Evaluating retrieval"):
        query = query_item["query"]
        golden_chunk_uuids = query_item["golden_chunk_uuids"]

        # Find all golden chunk contents
        golden_contents = []
        for doc_uuid, chunk_index in golden_chunk_uuids:
            golden_doc = next(
                (doc for doc in query_item["golden_documents"] if doc["uuid"] == doc_uuid), None
            )
            if not golden_doc:
                print(f"Warning: Golden document not found for UUID {doc_uuid}")
                continue

            golden_chunk = next(
                (chunk for chunk in golden_doc["chunks"] if chunk["index"] == chunk_index), None
            )
            if not golden_chunk:
                print(
                    f"Warning: Golden chunk not found for index {chunk_index} in document {doc_uuid}"
                )
                continue

            golden_contents.append(golden_chunk["content"].strip())

        if not golden_contents:
            print(f"Warning: No golden contents found for query: {query}")
            continue

        retrieved_docs = retrieval_function(query, db, k=k)

        # Count how many golden chunks are in the top k retrieved documents
        chunks_found = 0
        for golden_content in golden_contents:
            for doc in retrieved_docs[:k]:
                retrieved_content = (
                    doc["metadata"]
                    .get("original_content", doc["metadata"].get("content", ""))
                    .strip()
                )
                if retrieved_content == golden_content:
                    chunks_found += 1
                    break

        query_score = chunks_found / len(golden_contents)
        total_score += query_score

    average_score = total_score / total_queries
    pass_at_n = average_score * 100
    return {"pass_at_n": pass_at_n, "average_score": average_score, "total_queries": total_queries}


def retrieve_base(query: str, db, k: int = 20) -> list[dict[str, Any]]:
    """
    Retrieve relevant documents using either VectorDB or ContextualVectorDB.

    :param query: The query string
    :param db: The VectorDB or ContextualVectorDB instance
    :param k: Number of top results to retrieve
    :return: List of retrieved documents
    """
    return db.search(query, k=k)


def evaluate_db(db, original_jsonl_path: str, k):
    # Load the original JSONL data for queries and ground truth
    original_data = load_jsonl(original_jsonl_path)

    # Evaluate retrieval
    results = evaluate_retrieval(original_data, retrieve_base, db, k)
    return results


def evaluate_and_display(db, jsonl_path: str, k_values: list[int] = None, db_name: str = ""):
    """
    Evaluate retrieval performance across multiple k values and display formatted results.

    Args:
        db: Vector database instance (VectorDB or ContextualVectorDB)
        jsonl_path: Path to evaluation dataset
        k_values: List of k values to evaluate (default: [5, 10, 20])
        db_name: Optional name for the database being evaluated

    Returns:
        Dict mapping k values to their results
    """
    if k_values is None:
        k_values = [5, 10, 20]
    results = {}

    print(f"{'=' * 60}")
    if db_name:
        print(f"Evaluation Results: {db_name}")
    else:
        print("Evaluation Results")
    print(f"{'=' * 60}\n")

    for k in k_values:
        print(f"Evaluating Pass@{k}...")
        results[k] = evaluate_db(db, jsonl_path, k)
        print()  # Add spacing between evaluations

    # Print summary table
    print(f"{'=' * 60}")
    print(f"{'Metric':<15} {'Pass Rate':<15} {'Score':<15}")
    print(f"{'-' * 60}")
    for k in k_values:
        pass_rate = f"{results[k]['pass_at_n']:.2f}%"
        score = f"{results[k]['average_score']:.4f}"
        print(f"{'Pass@' + str(k):<15} {pass_rate:<15} {score:<15}")
    print(f"{'=' * 60}\n")

    return results

现在我们通过评估基础 RAG 系统来确立性能基线。我们在 k=5、10 和 20 处测试，看黄金块有多少出现在检索结果的前列。这为我们提供了衡量改进的基准。


In [ ]:
results = evaluate_and_display(
    base_db, "data/evaluation_set.jsonl", k_values=[5, 10, 20], db_name="Baseline RAG"
)

Evaluation Results: Contextual Embeddings

Evaluating Pass@5...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 65.26it/s]



Evaluating Pass@10...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.87it/s]



Evaluating Pass@20...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.72it/s]


Metric          Pass Rate       Score          
------------------------------------------------------------
Pass@5          80.92%          0.8092         
Pass@10         87.15%          0.8715         
Pass@20         90.06%          0.9006         



这些结果展示了我们的基线 RAG 性能。系统在 top 5 结果中正确检索到目标块的比率为 81%，在 top 10 中提升到 87%，在 top 20 中为 90%。

## 上下文嵌入向量

在基础 RAG 中，单个块单独嵌入时往往缺乏足够上下文。上下文嵌入向量通过让 Claude 生成一段简要描述来解决这一问题，把每个块“定位”在其源文档中。然后我们把块与这段上下文一起嵌入，形成更丰富的向量表示。

对于代码库数据集中的每个块，我们把该块及其完整源文件一起传给 Claude。Claude 生成一段简明说明，说明该块包含什么、在整篇文件中处于什么位置。这段上下文在嵌入之前被加到块的前面。


### 成本与时延考量

**这个成本发生在什么时候？** 上下文化只在摄取时执行一次，而不是每次查询都执行。与 HyDE（假设文档嵌入向量）这类会给每次检索增加时延的技术不同，上下文嵌入向量是构建向量数据库时的一次性成本。提示词缓存让这件事切实可行。由于我们按顺序处理同一文档的所有块，可以利用提示词缓存获得显著节省。

1. 第一个块：把整篇文档写入缓存（支付少量溢价）
2. 后续块：从缓存读取文档（这些词元享受 90% 折扣）
3. 缓存持续 5 分钟，足够处理完一篇文档中的所有块

**成本示例**：对于 8k 词元文档中的 800 词元块，加上 100 词元的生成上下文，总成本为每百万文档词元 $1.02。运行下面的代码时，你会在日志中看到缓存带来的节省。

**说明：** 有些嵌入模型有固定的输入词元上限。如果使用上下文嵌入向量后性能反而变差，你的上下文化块可能被截断了——建议改用上下文窗口更大的嵌入模型。

--- 

我们通过为单个块生成上下文，来看一个上下文嵌入向量如何工作的示例。我们用 Claude 生成定位上下文，同时你会看到提示词缓存指标的实际效果。


In [ ]:
DOCUMENT_CONTEXT_PROMPT = """
<document>
{doc_content}
</document>
"""

CHUNK_CONTEXT_PROMPT = """
Here is the chunk we want to situate within the whole document
<chunk>
{chunk_content}
</chunk>

Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else.
"""


def situate_context(doc: str, chunk: str) -> str:
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": DOCUMENT_CONTEXT_PROMPT.format(doc_content=doc),
                        "cache_control": {
                            "type": "ephemeral"
                        },  # we will make use of prompt caching for the full documents
                    },
                    {
                        "type": "text",
                        "text": CHUNK_CONTEXT_PROMPT.format(chunk_content=chunk),
                    },
                ],
            }
        ],
    )
    return response


jsonl_data = load_jsonl("data/evaluation_set.jsonl")
# Example usage
doc_content = jsonl_data[0]["golden_documents"][0]["content"]
chunk_content = jsonl_data[0]["golden_chunks"][0]["content"]

response = situate_context(doc_content, chunk_content)
print(f"Situated context: {response.content[0].text}")
print("-" * 10)
# Print cache performance metrics
print(f"Input tokens: {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")
print(f"Cache creation input tokens: {response.usage.cache_creation_input_tokens}")
print(f"Cache read input tokens: {response.usage.cache_read_input_tokens}")

Situated context: This chunk contains the module documentation and initial struct definition for a differential fuzzing executor. It introduces the `DiffExecutor` struct that wraps two executors (primary and secondary) to run them sequentially with the same input, comparing their behavior for differential testing. The chunk establishes the core data structure and imports needed for the differential fuzzing implementation.
----------
Input tokens: 3412
Output tokens: 76
Cache creation input tokens: 0
Cache read input tokens: 0


### 构建上下文向量数据库

既然已经看到如何为单个块生成上下文描述，我们把它扩展到整个数据集。下面的 `ContextualVectorDB` 类在基础 `VectorDB` 之上扩展，在摄取过程中自动进行上下文化。

**主要特性：**

- **并行处理**：使用 ThreadPoolExecutor 同时对多个块做上下文化（线程数可配置）
- **自动提示词缓存**：按文档逐个处理块，以最大化缓存命中
- **词元跟踪**：监控缓存性能并计算实际成本节省
- **持久化存储**：把嵌入向量和上下文化元数据一起保存到磁盘

运行这段代码时，请注意词元用量统计——你会看到 70-80% 的输入词元是从缓存读取的，这体现了提示词缓存带来的显著成本节省。在我们的 737 块数据集上，这把原本约 $15 的摄取任务降到约 $3。

In [ ]:
import json
import os
import pickle
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any

import anthropic
import numpy as np
import voyageai
from tqdm import tqdm


class ContextualVectorDB:
    def __init__(self, name: str, voyage_api_key=None, anthropic_api_key=None):
        if voyage_api_key is None:
            voyage_api_key = os.getenv("VOYAGE_API_KEY")
        if anthropic_api_key is None:
            anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

        self.voyage_client = voyageai.Client(api_key=voyage_api_key)
        self.anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)
        self.name = name
        self.embeddings = []
        self.metadata = []
        self.query_cache = {}
        self.db_path = f"./data/{name}/contextual_vector_db.pkl"

        self.token_counts = {"input": 0, "output": 0, "cache_read": 0, "cache_creation": 0}
        self.token_lock = threading.Lock()

    def situate_context(self, doc: str, chunk: str) -> tuple[str, Any]:
        DOCUMENT_CONTEXT_PROMPT = """
        <document>
        {doc_content}
        </document>
        """

        CHUNK_CONTEXT_PROMPT = """
        Here is the chunk we want to situate within the whole document
        <chunk>
        {chunk_content}
        </chunk>

        Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
        Answer only with the succinct context and nothing else.
        """

        response = self.anthropic_client.messages.create(
            model=MODEL_NAME,
            max_tokens=1000,
            temperature=0.0,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": DOCUMENT_CONTEXT_PROMPT.format(doc_content=doc),
                            "cache_control": {
                                "type": "ephemeral"
                            },  # we will make use of prompt caching for the full documents
                        },
                        {
                            "type": "text",
                            "text": CHUNK_CONTEXT_PROMPT.format(chunk_content=chunk),
                        },
                    ],
                },
            ],
            extra_headers={"anthropic-beta": "prompt-caching-2024-07-31"},
        )
        return response.content[0].text, response.usage

    def load_data(self, dataset: list[dict[str, Any]], parallel_threads: int = 1):
        if self.embeddings and self.metadata:
            print("Vector database is already loaded. Skipping data loading.")
            return
        if os.path.exists(self.db_path):
            print("Loading vector database from disk.")
            self.load_db()
            return

        texts_to_embed = []
        metadata = []
        total_chunks = sum(len(doc["chunks"]) for doc in dataset)

        def process_chunk(doc, chunk):
            # for each chunk, produce the context
            contextualized_text, usage = self.situate_context(doc["content"], chunk["content"])
            with self.token_lock:
                self.token_counts["input"] += usage.input_tokens
                self.token_counts["output"] += usage.output_tokens
                self.token_counts["cache_read"] += usage.cache_read_input_tokens
                self.token_counts["cache_creation"] += usage.cache_creation_input_tokens

            return {
                # append the context to the original text chunk
                "text_to_embed": f"{contextualized_text}\n\n{chunk['content']}",
                "metadata": {
                    "doc_id": doc["doc_id"],
                    "original_uuid": doc["original_uuid"],
                    "chunk_id": chunk["chunk_id"],
                    "original_index": chunk["original_index"],
                    "original_content": chunk["content"],
                    "contextualized_content": contextualized_text,
                },
            }

        print(f"Processing {total_chunks} chunks with {parallel_threads} threads")
        with ThreadPoolExecutor(max_workers=parallel_threads) as executor:
            futures = []
            for doc in dataset:
                for chunk in doc["chunks"]:
                    futures.append(executor.submit(process_chunk, doc, chunk))

            for future in tqdm(as_completed(futures), total=total_chunks, desc="Processing chunks"):
                result = future.result()
                texts_to_embed.append(result["text_to_embed"])
                metadata.append(result["metadata"])

        self._embed_and_store(texts_to_embed, metadata)
        self.save_db()

        # logging token usage
        print(
            f"Contextual Vector database loaded and saved. Total chunks processed: {len(texts_to_embed)}"
        )
        print(f"Total input tokens without caching: {self.token_counts['input']}")
        print(f"Total output tokens: {self.token_counts['output']}")
        print(f"Total input tokens written to cache: {self.token_counts['cache_creation']}")
        print(f"Total input tokens read from cache: {self.token_counts['cache_read']}")

        total_tokens = (
            self.token_counts["input"]
            + self.token_counts["cache_read"]
            + self.token_counts["cache_creation"]
        )
        savings_percentage = (
            (self.token_counts["cache_read"] / total_tokens) * 100 if total_tokens > 0 else 0
        )
        print(
            f"Total input token savings from prompt caching: {savings_percentage:.2f}% of all input tokens used were read from cache."
        )
        print("Tokens read from cache come at a 90 percent discount!")

    # we use voyage AI here for embeddings. Read more here: https://docs.voyageai.com/docs/embeddings
    def _embed_and_store(self, texts: list[str], data: list[dict[str, Any]]):
        batch_size = 128
        result = [
            self.voyage_client.embed(texts[i : i + batch_size], model="voyage-2").embeddings
            for i in range(0, len(texts), batch_size)
        ]
        self.embeddings = [embedding for batch in result for embedding in batch]
        self.metadata = data

    def search(self, query: str, k: int = 20) -> list[dict[str, Any]]:
        if query in self.query_cache:
            query_embedding = self.query_cache[query]
        else:
            query_embedding = self.voyage_client.embed([query], model="voyage-2").embeddings[0]
            self.query_cache[query] = query_embedding

        if not self.embeddings:
            raise ValueError("No data loaded in the vector database.")

        similarities = np.dot(self.embeddings, query_embedding)
        top_indices = np.argsort(similarities)[::-1][:k]

        top_results = []
        for idx in top_indices:
            result = {
                "metadata": self.metadata[idx],
                "similarity": float(similarities[idx]),
            }
            top_results.append(result)
        return top_results

    def save_db(self):
        data = {
            "embeddings": self.embeddings,
            "metadata": self.metadata,
            "query_cache": json.dumps(self.query_cache),
        }
        os.makedirs(os.path.dirname(self.db_path), exist_ok=True)
        with open(self.db_path, "wb") as file:
            pickle.dump(data, file)

    def load_db(self):
        if not os.path.exists(self.db_path):
            raise ValueError(
                "Vector database file not found. Use load_data to create a new database."
            )
        with open(self.db_path, "rb") as file:
            data = pickle.load(file)
        self.embeddings = data["embeddings"]
        self.metadata = data["metadata"]
        self.query_cache = json.loads(data["query_cache"])

In [22]:
# Load the transformed dataset
with open("data/codebase_chunks.json") as f:
    transformed_dataset = json.load(f)

# Initialize the ContextualVectorDB
contextual_db = ContextualVectorDB("my_contextual_db")

# Load and process the data
# note: consider increasing the number of parallel threads to run this faster, or reducing the number of parallel threads if concerned about hitting your API rate limit
contextual_db.load_data(transformed_dataset, parallel_threads=5)

Processing 737 chunks with 5 threads


Processing chunks: 100%|██████████| 737/737 [05:32<00:00,  2.22it/s]


Contextual Vector database loaded and saved. Total chunks processed: 737
Total input tokens without caching: 1223730
Total output tokens: 58161
Total input tokens written to cache: 176079
Total input tokens read from cache: 2267069
Total input token savings from prompt caching: 61.83% of all input tokens used were read from cache.
Tokens read from cache come at a 90 percent discount!


这些数字揭示了提示词缓存对上下文嵌入向量的作用：

- 我们处理了 **737 个块**，分布在 9 个代码库文件中
- **61.83% 的输入词元**从缓存读取（227 万词元享受 90% 折扣）
- 如果不使用缓存，仅输入词元就要花费 **约 $9.20**
- 使用缓存后，实际成本降到 **约 $2.85**（节省 69%）

缓存命中率取决于每个文档包含多少块。块更多的文件从缓存中获益更大，因为整篇文档只写入缓存一次，之后该文件的每个块都反复读取它。这就是为什么按文档顺序处理（而不是随机打乱块）对最大化缓存效率至关重要。

现在我们来评估，与基线相比，这种上下文化在多大程度上提升了检索性能。

In [28]:
results = evaluate_and_display(
    contextual_db,
    "data/evaluation_set.jsonl",
    k_values=[5, 10, 20],
    db_name="Contextual Embeddings",
)

Evaluation Results: Contextual Embeddings

Evaluating Pass@5...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.58it/s]



Evaluating Pass@10...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.37it/s]



Evaluating Pass@20...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.14it/s]


Metric          Pass Rate       Score          
------------------------------------------------------------
Pass@5          88.12%          0.8812         
Pass@10         92.34%          0.9234         
Pass@20         94.29%          0.9429         



通过在嵌入之前为每个块添加上下文，我们在所有 k 值上把检索失败减少了 **约 30-40%**。这意味着检索到的前列块中无关结果更少，当你把这些块交给 Claude 生成最终响应时，答案质量更好。

提升在 Pass@5 处最明显，而 Precision 在这里最为关键——这说明上下文化块不仅被检索到的次数更多，而且相关时排名更靠前。

## 上下文 BM25：混合检索

仅靠上下文嵌入向量，我们就把 Pass@10 从 87% 提升到 92%。把语义检索与基于关键词的检索结合，使用**上下文 BM25**可以进一步提升性能——这种混合方法能把检索失败率压得更低。

### 为什么用混合检索？

语义检索擅长理解含义和上下文，但可能漏掉精确的关键词匹配。BM25（一种概率性关键词排序算法）擅长查找特定词条，但缺乏语义理解。把两者结合，就能兼得两者之长：

- **语义检索**：捕捉概念相似性和转述表达
- **BM25**：捕捉精确术语、函数名和特定短语
- **倒数排名融合**：智能地合并两个来源的结果

### 什么是 BM25？

BM25 是一种概率性排序函数，它通过考虑文档长度和词项饱和度改进了 TF-IDF。它在生产搜索引擎（包括 Elasticsearch）中被广泛使用，因为在关键词相关性排序上效果出色。技术细节见[这篇博客文章](https://www.elastic.co/blog/practical-bm25-part-2-the-bm25-algorithm-and-its-variables)。

我们不只检索原始块内容，而是同时检索该块*以及*先前生成的上下文描述。这意味着 BM25 可以在原文或解释性上下文中匹配关键词。

### 环境准备：运行 Elasticsearch

运行下面的代码之前，你需要在本地运行 Elasticsearch。最简单的方式是用 Docker：

```bash
docker run -d --name elasticsearch -p 9200:9200 -p 9300:9300 \
  -e "discovery.type=single-node" \
  -e "xpack.security.enabled=false" \
  elasticsearch:9.2.0
```

## 故障排查：
- 确认它在运行：docker ps | grep elasticsearch
- 如果端口 9200 被占用：docker stop elasticsearch && docker rm elasticsearch
- 出现问题时查看日志：docker logs elasticsearch

## 混合检索如何工作

下面的 retrieve_advanced 函数实现了一个三步流程：

1. 检索候选：从语义检索和 BM25 各取前 150 个结果
2. 得分融合：用加权倒数排名融合合并两个排名
    - 默认：语义检索权重 80%，BM25 权重 20%
    - 这些权重可调——针对你的使用场景做实验以取得最优
3. 返回 top-k：选出融合后得分最高的结果

权重体系让你可以根据数据特点，在语义理解和关键词精确性之间取得平衡。

In [ ]:
import json
import os
from typing import Any

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from tqdm import tqdm


class ElasticsearchBM25:
    def __init__(self, index_name: str = "contextual_bm25_index"):
        self.es_client = Elasticsearch("http://localhost:9200")
        self.index_name = index_name
        self.create_index()

    def create_index(self):
        index_settings = {
            "settings": {
                "analysis": {"analyzer": {"default": {"type": "english"}}},
                "similarity": {"default": {"type": "BM25"}},
                "index.queries.cache.enabled": False,
            },
            "mappings": {
                "properties": {
                    "content": {"type": "text", "analyzer": "english"},
                    "contextualized_content": {"type": "text", "analyzer": "english"},
                    "doc_id": {"type": "keyword", "index": False},
                    "chunk_id": {"type": "keyword", "index": False},
                    "original_index": {"type": "integer", "index": False},
                }
            },
        }

        # Change this line - remove 'body=' parameter
        if not self.es_client.indices.exists(index=self.index_name):
            self.es_client.indices.create(
                index=self.index_name,
                settings=index_settings["settings"],
                mappings=index_settings["mappings"],
            )
            print(f"Created index: {self.index_name}")

    def index_documents(self, documents: list[dict[str, Any]]):
        actions = [
            {
                "_index": self.index_name,
                "_source": {
                    "content": doc["original_content"],
                    "contextualized_content": doc["contextualized_content"],
                    "doc_id": doc["doc_id"],
                    "chunk_id": doc["chunk_id"],
                    "original_index": doc["original_index"],
                },
            }
            for doc in documents
        ]
        success, _ = bulk(self.es_client, actions)
        self.es_client.indices.refresh(index=self.index_name)
        return success

    def search(self, query: str, k: int = 20) -> list[dict[str, Any]]:
        self.es_client.indices.refresh(index=self.index_name)

        # Change this - remove 'body=' and pass query directly
        response = self.es_client.search(
            index=self.index_name,
            query={
                "multi_match": {
                    "query": query,
                    "fields": ["content", "contextualized_content"],
                }
            },
            size=k,
        )

        return [
            {
                "doc_id": hit["_source"]["doc_id"],
                "original_index": hit["_source"]["original_index"],
                "content": hit["_source"]["content"],
                "contextualized_content": hit["_source"]["contextualized_content"],
                "score": hit["_score"],
            }
            for hit in response["hits"]["hits"]
        ]


def create_elasticsearch_bm25_index(db: ContextualVectorDB):
    es_bm25 = ElasticsearchBM25()
    es_bm25.index_documents(db.metadata)
    return es_bm25


def retrieve_advanced(
    query: str,
    db: ContextualVectorDB,
    es_bm25: ElasticsearchBM25,
    k: int,
    semantic_weight: float = 0.8,
    bm25_weight: float = 0.2,
):
    num_chunks_to_recall = 150

    # Semantic search
    semantic_results = db.search(query, k=num_chunks_to_recall)
    ranked_chunk_ids = [
        (result["metadata"]["doc_id"], result["metadata"]["original_index"])
        for result in semantic_results
    ]

    # BM25 search using Elasticsearch
    bm25_results = es_bm25.search(query, k=num_chunks_to_recall)
    ranked_bm25_chunk_ids = [
        (result["doc_id"], result["original_index"]) for result in bm25_results
    ]

    # Combine results
    chunk_ids = list(set(ranked_chunk_ids + ranked_bm25_chunk_ids))
    chunk_id_to_score = {}

    # Initial scoring with weights
    for chunk_id in chunk_ids:
        score = 0
        if chunk_id in ranked_chunk_ids:
            index = ranked_chunk_ids.index(chunk_id)
            score += semantic_weight * (1 / (index + 1))  # Weighted 1/n scoring for semantic
        if chunk_id in ranked_bm25_chunk_ids:
            index = ranked_bm25_chunk_ids.index(chunk_id)
            score += bm25_weight * (1 / (index + 1))  # Weighted 1/n scoring for BM25
        chunk_id_to_score[chunk_id] = score

    # Sort chunk IDs by their scores in descending order
    sorted_chunk_ids = sorted(
        chunk_id_to_score.keys(), key=lambda x: (chunk_id_to_score[x], x[0], x[1]), reverse=True
    )

    # Assign new scores based on the sorted order
    for index, chunk_id in enumerate(sorted_chunk_ids):
        chunk_id_to_score[chunk_id] = 1 / (index + 1)

    # Prepare the final results
    final_results = []
    semantic_count = 0
    bm25_count = 0
    for chunk_id in sorted_chunk_ids[:k]:
        chunk_metadata = next(
            chunk
            for chunk in db.metadata
            if chunk["doc_id"] == chunk_id[0] and chunk["original_index"] == chunk_id[1]
        )
        is_from_semantic = chunk_id in ranked_chunk_ids
        is_from_bm25 = chunk_id in ranked_bm25_chunk_ids
        final_results.append(
            {
                "chunk": chunk_metadata,
                "score": chunk_id_to_score[chunk_id],
                "from_semantic": is_from_semantic,
                "from_bm25": is_from_bm25,
            }
        )

        if is_from_semantic and not is_from_bm25:
            semantic_count += 1
        elif is_from_bm25 and not is_from_semantic:
            bm25_count += 1
        else:  # it's in both
            semantic_count += 0.5
            bm25_count += 0.5

    return final_results, semantic_count, bm25_count


def evaluate_db_advanced(
    db: ContextualVectorDB,
    original_jsonl_path: str,
    k_values: list[int] = None,
    db_name: str = "Hybrid Search",
):
    """
    Evaluate hybrid search (semantic + BM25) at multiple k values with formatted results.

    Args:
        db: ContextualVectorDB instance
        original_jsonl_path: Path to evaluation dataset
        k_values: List of k values to evaluate (default: [5, 10, 20])
        db_name: Name for the evaluation display

    Returns:
        Dict mapping k values to their results and source breakdowns
    """
    if k_values is None:
        k_values = [5, 10, 20]
    original_data = load_jsonl(original_jsonl_path)
    es_bm25 = create_elasticsearch_bm25_index(db)
    results = {}

    print(f"{'=' * 70}")
    print(f"Evaluation Results: {db_name}")
    print(f"{'=' * 70}\n")

    try:
        # Warm-up queries
        warm_up_queries = original_data[:10]
        for query_item in warm_up_queries:
            _ = retrieve_advanced(query_item["query"], db, es_bm25, k_values[0])

        for k in k_values:
            print(f"Evaluating Pass@{k}...")

            total_score = 0
            total_semantic_count = 0
            total_bm25_count = 0
            total_results = 0

            for query_item in tqdm(original_data, desc=f"Pass@{k}"):
                query = query_item["query"]
                golden_chunk_uuids = query_item["golden_chunk_uuids"]

                golden_contents = []
                for doc_uuid, chunk_index in golden_chunk_uuids:
                    golden_doc = next(
                        (doc for doc in query_item["golden_documents"] if doc["uuid"] == doc_uuid),
                        None,
                    )
                    if golden_doc:
                        golden_chunk = next(
                            (
                                chunk
                                for chunk in golden_doc["chunks"]
                                if chunk["index"] == chunk_index
                            ),
                            None,
                        )
                        if golden_chunk:
                            golden_contents.append(golden_chunk["content"].strip())

                if not golden_contents:
                    continue

                retrieved_docs, semantic_count, bm25_count = retrieve_advanced(
                    query, db, es_bm25, k
                )

                chunks_found = 0
                for golden_content in golden_contents:
                    for doc in retrieved_docs[:k]:
                        retrieved_content = doc["chunk"]["original_content"].strip()
                        if retrieved_content == golden_content:
                            chunks_found += 1
                            break

                query_score = chunks_found / len(golden_contents)
                total_score += query_score

                total_semantic_count += semantic_count
                total_bm25_count += bm25_count
                total_results += len(retrieved_docs)

            total_queries = len(original_data)
            average_score = total_score / total_queries
            pass_at_n = average_score * 100

            semantic_percentage = (
                (total_semantic_count / total_results) * 100 if total_results > 0 else 0
            )
            bm25_percentage = (total_bm25_count / total_results) * 100 if total_results > 0 else 0

            results[k] = {
                "pass_at_n": pass_at_n,
                "average_score": average_score,
                "total_queries": total_queries,
                "semantic_percentage": semantic_percentage,
                "bm25_percentage": bm25_percentage,
            }

            print(f"Pass@{k}: {pass_at_n:.2f}%")
            print(f"Semantic: {semantic_percentage:.1f}% | BM25: {bm25_percentage:.1f}%\n")

        # Print summary table
        print(f"{'=' * 70}")
        print(f"{'Metric':<12} {'Pass Rate':<12} {'Score':<12} {'Semantic':<12} {'BM25':<12}")
        print(f"{'-' * 70}")
        for k in k_values:
            r = results[k]
            print(
                f"{'Pass@' + str(k):<12} {r['pass_at_n']:>10.2f}% {r['average_score']:>10.4f} "
                f"{r['semantic_percentage']:>10.1f}% {r['bm25_percentage']:>10.1f}%"
            )
        print(f"{'=' * 70}\n")

        return results

    finally:
        # Delete the Elasticsearch index
        if es_bm25.es_client.indices.exists(index=es_bm25.index_name):
            es_bm25.es_client.indices.delete(index=es_bm25.index_name)
            print(f"Deleted Elasticsearch index: {es_bm25.index_name}")

In [39]:
results = evaluate_db_advanced(
    contextual_db,
    "data/evaluation_set.jsonl",
    k_values=[5, 10, 20],
    db_name="Contextual BM25 Hybrid Search",
)

Created index: contextual_bm25_index
Evaluation Results: Contextual BM25 Hybrid Search

Evaluating Pass@5...


Pass@5: 100%|██████████| 248/248 [00:05<00:00, 41.79it/s]


Pass@5: 88.86%
Semantic: 54.6% | BM25: 45.4%

Evaluating Pass@10...


Pass@10: 100%|██████████| 248/248 [00:05<00:00, 42.20it/s]


Pass@10: 92.31%
Semantic: 57.6% | BM25: 42.4%

Evaluating Pass@20...


Pass@20: 100%|██████████| 248/248 [00:05<00:00, 42.15it/s]


Pass@20: 95.23%
Semantic: 60.8% | BM25: 39.2%

Metric       Pass Rate    Score        Semantic     BM25        
----------------------------------------------------------------------
Pass@5            88.86%     0.8886       54.6%       45.4%
Pass@10           92.31%     0.9231       57.6%       42.4%
Pass@20           95.23%     0.9523       60.8%       39.2%

Deleted Elasticsearch index: contextual_bm25_index


## 重排序

混合检索已经取得了很强的结果（Pass@10 为 93.21%），但还有一项技术可以榨出更多性能：**重排序**。

### 什么是重排序？

重排序是一种两阶段检索方法：

1. **阶段 1 - 广检索**：撒大网，检索比所需更多的候选（例如检索 100 个块）
2. **阶段 2 - 精确挑选**：用专门的重排序模型给这些候选打分，只选出得分最高的 top-k 个最相关项

**为什么有效？** 初始检索方法（嵌入向量、BM25）针对数百万文档的速度做了优化。重排序模型更慢但更准确——它们可以在较小的候选集上做更深入的分析。这种速度/准确率的权衡在实践中效果很好。

### 我们的重排序方法

在这个示例中，我们使用一个更简单的重排序工作流，它只建立在上下文嵌入向量之上（不使用完整混合检索）。流程如下：

1. **超额检索**：检索比所需多 10 倍的结果（例如需要 10 个时检索 100 个块）
2. **用 Cohere 重排序**：使用 Cohere 的 `rerank-english-v3.0` 模型给全部候选打分
3. **挑选 top-k**：只返回得分最高的结果

重排序模型能够同时看到原始块内容和我们生成的上下文描述，这为它做出精确的相关性判断提供了丰富信息。

### 预期性能

加入重排序带来适度但有意义的提升：
- **不重排序**：Pass@10 为 92.34%（仅用上下文嵌入向量）
- **重排序**：Pass@10 约 95%（额外提升 2-3%）

这看起来不大，但在生产系统中，把失败率从 7.66% 降到约 5% 能显著改善用户体验。代价是查询时延——重排序每次查询增加约 100-200 毫秒，具体取决于候选集大小。

In [ ]:
import json
from collections.abc import Callable
from typing import Any

import cohere
from tqdm import tqdm


def evaluate_db_rerank(
    db, original_jsonl_path: str, k_values: list[int] = None, db_name: str = "Reranking"
):
    """
    Evaluate reranking performance at multiple k values with formatted results.

    Args:
        db: ContextualVectorDB instance
        original_jsonl_path: Path to evaluation dataset
        k_values: List of k values to evaluate (default: [5, 10, 20])
        db_name: Name for the evaluation display

    Returns:
        Dict mapping k values to their results
    """
    if k_values is None:
        k_values = [5, 10, 20]
    original_data = load_jsonl(original_jsonl_path)
    co = cohere.Client(os.getenv("COHERE_API_KEY"))
    results = {}

    print(f"{'=' * 60}")
    print(f"Evaluation Results: {db_name}")
    print(f"{'=' * 60}\n")

    for k in k_values:
        print(f"Evaluating Pass@{k} with reranking...")

        total_score = 0
        total_queries = len(original_data)

        for query_item in tqdm(original_data, desc=f"Pass@{k}"):
            query = query_item["query"]
            golden_chunk_uuids = query_item["golden_chunk_uuids"]

            # Find golden contents
            golden_contents = []
            for doc_uuid, chunk_index in golden_chunk_uuids:
                golden_doc = next(
                    (doc for doc in query_item["golden_documents"] if doc["uuid"] == doc_uuid), None
                )
                if golden_doc:
                    golden_chunk = next(
                        (chunk for chunk in golden_doc["chunks"] if chunk["index"] == chunk_index),
                        None,
                    )
                    if golden_chunk:
                        golden_contents.append(golden_chunk["content"].strip())

            if not golden_contents:
                continue

            # Retrieve and rerank
            semantic_results = db.search(query, k=k * 10)

            # Prepare documents for reranking
            documents = [
                f"{res['metadata']['original_content']}\n\nContext: {res['metadata']['contextualized_content']}"
                for res in semantic_results
            ]

            # Rerank
            rerank_response = co.rerank(
                model="rerank-english-v3.0", query=query, documents=documents, top_n=k
            )
            time.sleep(0.1)  # Rate limiting

            # Get final results
            retrieved_docs = []
            for r in rerank_response.results:
                original_result = semantic_results[r.index]
                retrieved_docs.append(
                    {"chunk": original_result["metadata"], "score": r.relevance_score}
                )

            # Check if golden chunks are in results
            chunks_found = 0
            for golden_content in golden_contents:
                for doc in retrieved_docs[:k]:
                    retrieved_content = doc["chunk"]["original_content"].strip()
                    if retrieved_content == golden_content:
                        chunks_found += 1
                        break

            query_score = chunks_found / len(golden_contents)
            total_score += query_score

        average_score = total_score / total_queries
        pass_at_n = average_score * 100

        results[k] = {
            "pass_at_n": pass_at_n,
            "average_score": average_score,
            "total_queries": total_queries,
        }

        print(f"Pass@{k}: {pass_at_n:.2f}%")
        print(f"Average Score: {average_score:.4f}\n")

    # Print summary table
    print(f"{'=' * 60}")
    print(f"{'Metric':<15} {'Pass Rate':<15} {'Score':<15}")
    print(f"{'-' * 60}")
    for k in k_values:
        pass_rate = f"{results[k]['pass_at_n']:.2f}%"
        score = f"{results[k]['average_score']:.4f}"
        print(f"{'Pass@' + str(k):<15} {pass_rate:<15} {score:<15}")
    print(f"{'=' * 60}\n")

    return results

In [48]:
results = evaluate_db_rerank(
    contextual_db,
    "data/evaluation_set.jsonl",
    k_values=[5, 10, 20],
    db_name="Contextual Embeddings + Reranking",
)

Evaluation Results: Contextual Embeddings + Reranking

Evaluating Pass@5 with reranking...


Pass@5: 100%|██████████| 248/248 [01:40<00:00,  2.47it/s]


Pass@5: 92.15%
Average Score: 0.9215

Evaluating Pass@10 with reranking...


Pass@10: 100%|██████████| 248/248 [02:29<00:00,  1.66it/s]


Pass@10: 95.26%
Average Score: 0.9526

Evaluating Pass@20 with reranking...


Pass@20: 100%|██████████| 248/248 [03:03<00:00,  1.35it/s]

Pass@20: 97.45%
Average Score: 0.9745

Metric          Pass Rate       Score          
------------------------------------------------------------
Pass@5          92.15%          0.9215         
Pass@10         95.26%          0.9526         
Pass@20         97.45%          0.9745         




重排序带来了我们最强的结果，几乎消除了检索失败。我们来看看每项技术如何在上一项基础上叠加，从而实现这一提升。

从 Pass@10 为 87% 的基线 RAG 系统出发，通过系统性地应用先进检索技术，我们提升到了 95% 以上。每种方法解决不同的弱点：上下文嵌入向量解决“孤立块”问题，混合检索捕捉嵌入向量漏掉的关键词型查询，重排序则用更精细的相关性打分来优化最终挑选。

| 方法 | Pass@5 | Pass@10 | Pass@20 |
|----------|--------|---------|---------|
| **基线 RAG** | 80.92% | 87.15% | 90.06% |
| **+ 上下文嵌入向量** | 88.12% | 92.34% | 94.29% |
| **+ 混合检索（BM25）** | 86.43% | 93.21% | 94.99% |
| **+ 重排序** | 92.15% | 95.26% | 97.45% |

**关键要点：**

1. **上下文嵌入向量带来了最大的单项提升**（提高 5-7 个百分点），这印证了为块添加文档级上下文能显著改善检索质量。仅这一项技术就能让你达到最优性能的 90%。

2. **重排序取得了最高的绝对性能**，Pass@10 达到 95.26%——也就是说 95% 的查询中，正确块出现在前 10 个结果里。与基线 RAG 相比，这相当于**检索失败减少 47%**（失败率从 12.85% 降到 4.74%）。

3. **权衡很重要**：每项技术都会增加复杂性和成本：
   - 上下文嵌入向量：一次性摄取成本（本数据集配合提示词缓存约 $3）
   - 混合检索：需要 Elasticsearch 基础设施及其维护
   - 重排序：每次查询增加 100-200 毫秒时延，并有按查询计费的 API 成本（每次查询约 $0.002）

4. **根据你的需求选择方案**：
   - **高并发、成本敏感**：只用上下文嵌入向量（Pass@10 为 92%，无按查询成本）
   - **追求最高准确率、可容忍时延**：完整重排序工作流（Pass@10 为 95%，Precision 最佳）
   - **均衡的生产系统**：混合检索在不产生按查询成本的前提下提供强劲性能（Pass@10 为 93%）

对大多数生产 RAG 系统而言，**上下文嵌入向量的性能成本比最佳**，只需一次性摄取成本就能达到 Pass@10 为 92%。当你需要额外的 2-3 个百分点准确率并且能承担相应基础设施或查询成本时，可以再使用混合检索和重排序。

### 后续步骤与关键要点

1）我们演示了如何用上下文嵌入向量提升检索性能，随后又用上下文 BM25 和重排序带来了额外提升。

2）本示例使用的是代码库，但这些方法同样适用于其他数据类型，例如公司内部知识库、财务与法律内容、教育内容等等。

3）如果你是 AWS 用户，可以从 `contextual-rag-lambda-function` 中的 Lambda 函数开始上手；如果你是 GCP 用户，可以自行启动一个 Cloud Run 实例并采用类似的模式！